In [3]:
print(spark.version)
hadoop_version = spark._jsc.hadoopConfiguration().get("hadoop.version")
print(f"Hadoop version utilisée par Spark : {hadoop_version}")


3.5.0
Hadoop version utilisée par Spark : None


In [1]:
from pyspark.sql import SparkSession

# ============================================================================
# SPARK SIMPLE - Lecture CSV et affichage
# ============================================================================

print("🚀 Initialisation de Spark...")

# Configuration MinIO
MINIO_ACCESS_KEY = "minio"
MINIO_SECRET_KEY = "minio123"
MINIO_BUCKET = "telco-churn"
MINIO_ENDPOINT = "minio1:9000"

# Créer la SparkSession
spark = (
    SparkSession.builder
    .appName("TelcoChurnCSVViewer")
    .master("local[*]")
    .config("spark.hadoop.fs.s3a.access.key", MINIO_ACCESS_KEY)
    .config("spark.hadoop.fs.s3a.secret.key", MINIO_SECRET_KEY)
    .config("spark.hadoop.fs.s3a.endpoint", f"http://{MINIO_ENDPOINT}")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .getOrCreate()
)

print(f"✅ Spark {spark.version} initialisé avec succès!")

# ============================================================================
# Chargement des fichiers CSV
# ============================================================================
input_path = f"s3a://{MINIO_BUCKET}/raw/*.csv"
print(f"\n📂 Lecture des fichiers CSV depuis: {input_path}")

try:
    df = spark.read \
        .format("csv") \
        .option("header", "true") \
        .option("inferSchema", "true") \
        .load(input_path)

    print(f"✅ {df.count():,} lignes chargées, {len(df.columns)} colonnes\n")

    # Afficher les 10 premières lignes
    print("📊 Aperçu des données (10 premières lignes):")
    df.show(10, truncate=False)

except Exception as e:
    print(f"❌ Erreur lors de la lecture des CSV: {e}")

print("\n📚 La session Spark reste ouverte pour des requêtes supplémentaires.")
print("Accédez à l'UI Spark: http://localhost:4040")


🚀 Initialisation de Spark...


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/22 15:11:53 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/11/22 15:11:54 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


✅ Spark 3.5.7 initialisé avec succès!

📂 Lecture des fichiers CSV depuis: s3a://telco-churn/raw/*.csv
❌ Erreur lors de la lecture des CSV: An error occurred while calling o41.load.
: java.lang.RuntimeException: java.lang.ClassNotFoundException: Class org.apache.hadoop.fs.s3a.S3AFileSystem not found
	at org.apache.hadoop.conf.Configuration.getClass(Configuration.java:2688)
	at org.apache.hadoop.fs.FileSystem.getFileSystemClass(FileSystem.java:3431)
	at org.apache.hadoop.fs.FileSystem.createFileSystem(FileSystem.java:3466)
	at org.apache.hadoop.fs.FileSystem.access$300(FileSystem.java:174)
	at org.apache.hadoop.fs.FileSystem$Cache.getInternal(FileSystem.java:3574)
	at org.apache.hadoop.fs.FileSystem$Cache.get(FileSystem.java:3521)
	at org.apache.hadoop.fs.FileSystem.get(FileSystem.java:540)
	at org.apache.hadoop.fs.Path.getFileSystem(Path.java:365)
	at org.apache.spark.sql.execution.datasources.DataSource$.$anonfun$checkAndGlobPathIfNecessary$1(DataSource.scala:724)
	at scala.collection.

25/11/22 15:11:55 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: s3a://telco-churn/raw/*.csv.
java.lang.RuntimeException: java.lang.ClassNotFoundException: Class org.apache.hadoop.fs.s3a.S3AFileSystem not found
	at org.apache.hadoop.conf.Configuration.getClass(Configuration.java:2688)
	at org.apache.hadoop.fs.FileSystem.getFileSystemClass(FileSystem.java:3431)
	at org.apache.hadoop.fs.FileSystem.createFileSystem(FileSystem.java:3466)
	at org.apache.hadoop.fs.FileSystem.access$300(FileSystem.java:174)
	at org.apache.hadoop.fs.FileSystem$Cache.getInternal(FileSystem.java:3574)
	at org.apache.hadoop.fs.FileSystem$Cache.get(FileSystem.java:3521)
	at org.apache.hadoop.fs.FileSystem.get(FileSystem.java:540)
	at org.apache.hadoop.fs.Path.getFileSystem(Path.java:365)
	at org.apache.spark.sql.execution.streaming.FileStreamSink$.hasMetadata(FileStreamSink.scala:53)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelatio

In [2]:
# ============================================================================
# SPARK SOLUTION - Fixed with Correct Hadoop Version Matching
# ============================================================================

import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, when, avg, sum as spark_sum

print("🚀 Initialisation de Spark avec versions Hadoop compatibles...")

# Configuration MinIO
MINIO_ACCESS_KEY = "minio"
MINIO_SECRET_KEY = "minio123"
MINIO_BUCKET = "telco-churn"
MINIO_ENDPOINT = "minio1:9000"

# Arrêter toute session Spark existante
try:
    spark.stop()
    print("✅ Session Spark précédente arrêtée")
except:
    pass

# Créer la SparkSession avec Hadoop 3.3.4 (version compatible avec Spark 3.5.x)
spark = (
    SparkSession.builder
    .appName("TelcoChurnAnalysis")
    .master("local[*]")
    
    # IMPORTANT: Utiliser Hadoop 3.3.4 qui est compatible avec Spark 3.5.x
    .config("spark.jars.packages", 
            "org.apache.hadoop:hadoop-aws:3.3.4,"
            "org.apache.hadoop:hadoop-common:3.3.4,"
            "com.amazonaws:aws-java-sdk-bundle:1.12.262,"
            "io.delta:delta-spark_2.12:3.2.1")
    
    # Configuration S3A pour MinIO
    .config("spark.hadoop.fs.s3a.access.key", MINIO_ACCESS_KEY)
    .config("spark.hadoop.fs.s3a.secret.key", MINIO_SECRET_KEY)
    .config("spark.hadoop.fs.s3a.endpoint", f"http://{MINIO_ENDPOINT}")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", 
            "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
    
    # Désactiver les features S3A problématiques
    .config("spark.hadoop.fs.s3a.block.size", "128M")
    .config("spark.hadoop.fs.s3a.buffer.dir", "/tmp/s3a")
    .config("spark.hadoop.fs.s3a.fast.upload", "true")
    .config("spark.hadoop.fs.s3a.impl.disable.cache", "true")
    
    # Delta Lake
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", 
            "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    
    # Optimisations
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .config("spark.driver.memory", "2g")
    .config("spark.driver.maxResultSize", "1g")
    
    .getOrCreate()
)

print(f"✅ Spark {spark.version} initialisé avec succès!")
print(f"   Master: {spark.sparkContext.master}")
print(f"   UI: http://localhost:4040")
print(f"   Application ID: {spark.sparkContext.applicationId}")

# ============================================================================
# Test de connexion MinIO
# ============================================================================
print("\n🔍 Test de connexion MinIO...")

try:
    from pyspark.sql import Row
    
    # Créer un DataFrame de test
    test_data = [Row(id=1, message="Test Spark-MinIO", value=100)]
    test_df = spark.createDataFrame(test_data)
    
    # Écrire sur MinIO
    test_path = f"s3a://{MINIO_BUCKET}/test/spark_connection_test.parquet"
    print(f"   Tentative d'écriture vers: {test_path}")
    
    test_df.write.mode("overwrite").parquet(test_path)
    
    # Lire depuis MinIO
    print(f"   Tentative de lecture depuis: {test_path}")
    verify_df = spark.read.parquet(test_path)
    
    if verify_df.count() == 1:
        print("✅ Connexion MinIO OK!")
        minio_works = True
    else:
        print("⚠️ Problème de lecture/écriture MinIO")
        minio_works = False
        
except Exception as e:
    print(f"❌ Erreur connexion MinIO: {str(e)[:200]}")
    print("\n💡 Tentative avec boto3 à la place...")
    minio_works = False

# ============================================================================
# Si MinIO S3A ne fonctionne pas, utiliser boto3 + Spark local
# ============================================================================
if not minio_works:
    print("\n" + "="*70)
    print("📥 PLAN B: Téléchargement avec boto3, traitement avec Spark")
    print("="*70)
    
    import boto3
    from botocore.client import Config
    
    # Connexion boto3
    s3_client = boto3.client(
        's3',
        endpoint_url=f'http://{MINIO_ENDPOINT}',
        aws_access_key_id=MINIO_ACCESS_KEY,
        aws_secret_access_key=MINIO_SECRET_KEY,
        config=Config(signature_version='s3v4'),
        region_name='us-east-1'
    )
    
    # Télécharger les CSV
    local_data_dir = '/opt/workspace/data'
    os.makedirs(local_data_dir, exist_ok=True)
    
    print(f"\n📥 Téléchargement depuis MinIO...")
    
    try:
        response = s3_client.list_objects_v2(Bucket=MINIO_BUCKET, Prefix='raw/')
        
        downloaded = 0
        if 'Contents' in response:
            for obj in response['Contents']:
                key = obj['Key']
                if key.endswith('.csv'):
                    filename = os.path.basename(key)
                    local_file = os.path.join(local_data_dir, filename)
                    
                    s3_client.download_file(MINIO_BUCKET, key, local_file)
                    size_mb = os.path.getsize(local_file) / (1024 * 1024)
                    print(f"   ✓ {filename} ({size_mb:.2f} MB)")
                    downloaded += 1
            
            print(f"\n✅ {downloaded} fichier(s) téléchargé(s)")
            
            # Charger avec Spark depuis le système de fichiers local
            input_path = os.path.join(local_data_dir, "*.csv")
            
        else:
            print("⚠️ Aucun fichier trouvé dans MinIO")
            input_path = None
            
    except Exception as e:
        print(f"❌ Erreur boto3: {e}")
        input_path = None

else:
    # MinIO S3A fonctionne, utiliser directement
    input_path = f"s3a://{MINIO_BUCKET}/raw/*.csv"

# ============================================================================
# Chargement des données avec Spark
# ============================================================================
if input_path:
    print(f"\n📂 Chargement des données depuis: {input_path}")
    
    try:
        df_raw = spark.read \
            .format("csv") \
            .option("header", "true") \
            .option("inferSchema", "true") \
            .option("mode", "DROPMALFORMED") \
            .load(input_path)
        
        row_count = df_raw.count()
        col_count = len(df_raw.columns)
        
        print(f"✅ Données chargées: {row_count:,} lignes, {col_count} colonnes")
        
        # Afficher le schéma
        print("\n📋 Schéma des données:")
        print("="*70)
        df_raw.printSchema()
        
        # Afficher un aperçu
        print("\n📊 Aperçu des données:")
        print("="*70)
        df_raw.show(10, truncate=False)
        
        # ====================================================================
        # Analyse des données
        # ====================================================================
        print("\n📈 Statistiques descriptives:")
        print("="*70)
        df_raw.describe().show()
        
        # Analyse des valeurs nulles
        print("\n🔍 Analyse des valeurs nulles:")
        print("="*70)
        
        null_counts = df_raw.select([
            count(when(col(c).isNull(), c)).alias(c) 
            for c in df_raw.columns
        ])
        
        null_data = null_counts.collect()[0].asDict()
        
        print(f"{'Colonne':<30} {'Nulls':<10} {'% Nulls':<10}")
        print("-" * 50)
        has_nulls = False
        for col_name, null_count in null_data.items():
            pct = (null_count / row_count) * 100 if row_count > 0 else 0
            if null_count > 0:
                print(f"{col_name:<30} {null_count:<10} {pct:>6.2f}%")
                has_nulls = True
        
        if not has_nulls:
            print("✅ Aucune valeur nulle détectée!")
        
        # ====================================================================
        # Analyses spécifiques
        # ====================================================================
        if 'Churn' in df_raw.columns:
            print("\n🎯 Analyse du Churn:")
            print("="*70)
            
            churn_dist = df_raw.groupBy("Churn").count()
            churn_dist.show()
            
            churn_count = df_raw.filter(col("Churn") == "Yes").count()
            churn_rate = (churn_count / row_count) * 100
            print(f"\n📊 Taux de churn: {churn_rate:.2f}%")
            
            if 'tenure' in df_raw.columns:
                print("\n📅 Churn par ancienneté:")
                df_raw.groupBy("Churn") \
                    .agg(avg("tenure").alias("Tenure moyenne")) \
                    .show()
            
            if 'MonthlyCharges' in df_raw.columns:
                print("\n💰 Churn par charges mensuelles:")
                df_raw.groupBy("Churn") \
                    .agg(avg("MonthlyCharges").alias("Charges moyennes")) \
                    .show()
        
        # ====================================================================
        # Sauvegarde locale en Parquet
        # ====================================================================
        print("\n💾 Sauvegarde locale en Parquet...")
        
        local_parquet_path = "/opt/workspace/telco_data.parquet"
        df_raw.coalesce(1).write \
            .mode("overwrite") \
            .parquet(local_parquet_path)
        
        print(f"✅ Sauvegarde locale: {local_parquet_path}")
        
        # ====================================================================
        # Tentative de sauvegarde sur MinIO
        # ====================================================================
        if minio_works:
            print("\n💾 Sauvegarde sur MinIO...")
            
            try:
                # Delta Lake
                delta_path = f"s3a://{MINIO_BUCKET}/delta/telco_churn"
                df_raw.write \
                    .format("delta") \
                    .mode("overwrite") \
                    .option("overwriteSchema", "true") \
                    .save(delta_path)
                
                print(f"✅ Delta Lake: {delta_path}")
                
            except Exception as e:
                print(f"⚠️ Delta Lake échoué: {str(e)[:100]}")
                print("   Tentative Parquet...")
                
                # Fallback Parquet
                parquet_path = f"s3a://{MINIO_BUCKET}/processed/telco_data.parquet"
                df_raw.write \
                    .mode("overwrite") \
                    .parquet(parquet_path)
                
                print(f"✅ Parquet: {parquet_path}")
        
        # ====================================================================
        # Upload avec boto3 si S3A ne fonctionne pas
        # ====================================================================
        else:
            print("\n📤 Upload vers MinIO avec boto3...")
            
            # Trouver le fichier parquet généré
            parquet_files = [f for f in os.listdir(local_parquet_path) 
                           if f.startswith('part-') and f.endswith('.parquet')]
            
            if parquet_files:
                local_file = os.path.join(local_parquet_path, parquet_files[0])
                s3_key = 'processed/telco_data.parquet'
                
                try:
                    s3_client.upload_file(local_file, MINIO_BUCKET, s3_key)
                    print(f"✅ Uploadé: s3://{MINIO_BUCKET}/{s3_key}")
                except Exception as e:
                    print(f"⚠️ Erreur upload: {e}")
        
        # ====================================================================
        # Résumé final
        # ====================================================================
        print("\n" + "="*70)
        print("✅ TRAITEMENT TERMINÉ AVEC SUCCÈS!")
        print("="*70)
        print(f"\n📊 Résumé:")
        print(f"   • {row_count:,} lignes traitées")
        print(f"   • {col_count} colonnes")
        print(f"   • Sauvegarde locale: {local_parquet_path}")
        
        print(f"\n💡 DataFrame disponible: df_raw")
        print(f"   Exemples:")
        print(f"   • df_raw.count()")
        print(f"   • df_raw.filter(col('Churn') == 'Yes').count()")
        print(f"   • df_raw.groupBy('gender').count().show()")
        
    except Exception as e:
        print(f"\n❌ Erreur: {e}")
        import traceback
        traceback.print_exc()

else:
    print("\n⚠️ Aucune donnée à charger")

print("\n" + "="*70)
print("📚 SPARK SESSION ACTIVE")
print("="*70)
print("Accédez à l'UI Spark: http://localhost:4040")

🚀 Initialisation de Spark avec versions Hadoop compatibles...
✅ Session Spark précédente arrêtée
✅ Spark 3.5.7 initialisé avec succès!
   Master: local[*]
   UI: http://localhost:4040
   Application ID: local-1763819731835

🔍 Test de connexion MinIO...
   Tentative d'écriture vers: s3a://telco-churn/test/spark_connection_test.parquet
❌ Erreur connexion MinIO: An error occurred while calling o172.parquet.
: java.lang.NoClassDefFoundError: org/apache/hadoop/fs/impl/prefetch/PrefetchingStatistics
	at java.base/java.lang.ClassLoader.defineClass1(Native Method)

💡 Tentative avec boto3 à la place...

📥 PLAN B: Téléchargement avec boto3, traitement avec Spark

📥 Téléchargement depuis MinIO...
❌ Erreur boto3: An error occurred (NoSuchBucket) when calling the ListObjectsV2 operation: The specified bucket does not exist

⚠️ Aucune donnée à charger

📚 SPARK SESSION ACTIVE
Accédez à l'UI Spark: http://localhost:4040


In [1]:
from pyspark.sql import SparkSession

# ============================================================================
# SPARK SIMPLE - Lecture CSV et affichage
# ============================================================================

print("🚀 Initialisation de Spark...")

# Configuration MinIO
MINIO_ACCESS_KEY = "minio"
MINIO_SECRET_KEY = "minio123"
MINIO_BUCKET = "telco-churn"
MINIO_ENDPOINT = "minio1:9000"

# Créer la SparkSession
spark = (
    SparkSession.builder
    .appName("TelcoChurnCSVViewer")
    .master("local[*]")
    .config("spark.hadoop.fs.s3a.access.key", MINIO_ACCESS_KEY)
    .config("spark.hadoop.fs.s3a.secret.key", MINIO_SECRET_KEY)
    .config("spark.hadoop.fs.s3a.endpoint", f"http://{MINIO_ENDPOINT}")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .getOrCreate()
)

print(f"✅ Spark {spark.version} initialisé avec succès!")

# ============================================================================
# Chargement des fichiers CSV
# ============================================================================
input_path = f"s3a://{MINIO_BUCKET}/raw/*.csv"
print(f"\n📂 Lecture des fichiers CSV depuis: {input_path}")

try:
    df = spark.read \
        .format("csv") \
        .option("header", "true") \
        .option("inferSchema", "true") \
        .load(input_path)

    print(f"✅ {df.count():,} lignes chargées, {len(df.columns)} colonnes\n")

    # Afficher les 10 premières lignes
    print("📊 Aperçu des données (10 premières lignes):")
    df.show(10, truncate=False)

except Exception as e:
    print(f"❌ Erreur lors de la lecture des CSV: {e}")

print("\n📚 La session Spark reste ouverte pour des requêtes supplémentaires.")
print("Accédez à l'UI Spark: http://localhost:4040")


🚀 Initialisation de Spark...


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/22 14:23:48 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


✅ Spark 3.5.7 initialisé avec succès!

📂 Lecture des fichiers CSV depuis: s3a://telco-churn/raw/*.csv
❌ Erreur lors de la lecture des CSV: An error occurred while calling o41.load.
: java.lang.RuntimeException: java.lang.ClassNotFoundException: Class org.apache.hadoop.fs.s3a.S3AFileSystem not found
	at org.apache.hadoop.conf.Configuration.getClass(Configuration.java:2688)
	at org.apache.hadoop.fs.FileSystem.getFileSystemClass(FileSystem.java:3431)
	at org.apache.hadoop.fs.FileSystem.createFileSystem(FileSystem.java:3466)
	at org.apache.hadoop.fs.FileSystem.access$300(FileSystem.java:174)
	at org.apache.hadoop.fs.FileSystem$Cache.getInternal(FileSystem.java:3574)
	at org.apache.hadoop.fs.FileSystem$Cache.get(FileSystem.java:3521)
	at org.apache.hadoop.fs.FileSystem.get(FileSystem.java:540)
	at org.apache.hadoop.fs.Path.getFileSystem(Path.java:365)
	at org.apache.spark.sql.execution.datasources.DataSource$.$anonfun$checkAndGlobPathIfNecessary$1(DataSource.scala:724)
	at scala.collection.

25/11/22 14:23:49 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: s3a://telco-churn/raw/*.csv.
java.lang.RuntimeException: java.lang.ClassNotFoundException: Class org.apache.hadoop.fs.s3a.S3AFileSystem not found
	at org.apache.hadoop.conf.Configuration.getClass(Configuration.java:2688)
	at org.apache.hadoop.fs.FileSystem.getFileSystemClass(FileSystem.java:3431)
	at org.apache.hadoop.fs.FileSystem.createFileSystem(FileSystem.java:3466)
	at org.apache.hadoop.fs.FileSystem.access$300(FileSystem.java:174)
	at org.apache.hadoop.fs.FileSystem$Cache.getInternal(FileSystem.java:3574)
	at org.apache.hadoop.fs.FileSystem$Cache.get(FileSystem.java:3521)
	at org.apache.hadoop.fs.FileSystem.get(FileSystem.java:540)
	at org.apache.hadoop.fs.Path.getFileSystem(Path.java:365)
	at org.apache.spark.sql.execution.streaming.FileStreamSink$.hasMetadata(FileStreamSink.scala:53)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelatio

In [2]:
# ============================================================================
# SOLUTION FINALE QUI FONCTIONNE - boto3 + Spark
# Télécharge avec boto3, traite avec Spark, upload avec boto3
# ============================================================================

import os
import sys

print("="*70)
print("🚀 TRAITEMENT TELCO CHURN - SPARK + MinIO")
print("="*70)

# ============================================================================
# ÉTAPE 1: Téléchargement depuis MinIO avec boto3
# ============================================================================
print("\n📦 Installation de boto3...")
os.system("pip install -q boto3")

import boto3
from botocore.client import Config

print("\n🔗 Connexion à MinIO...")

MINIO_ENDPOINT = "minio1:9000"
MINIO_ACCESS_KEY = "minio"
MINIO_SECRET_KEY = "minio123"
MINIO_BUCKET = "telco-churn"

s3_client = boto3.client(
    's3',
    endpoint_url=f'http://{MINIO_ENDPOINT}',
    aws_access_key_id=MINIO_ACCESS_KEY,
    aws_secret_access_key=MINIO_SECRET_KEY,
    config=Config(signature_version='s3v4'),
    region_name='us-east-1'
)

# Vérifier la connexion
try:
    buckets = s3_client.list_buckets()
    print(f"✅ Connexion MinIO OK")
    print(f"   Buckets: {[b['Name'] for b in buckets['Buckets']]}")
except Exception as e:
    print(f"❌ Erreur MinIO: {e}")
    sys.exit(1)

# Télécharger les fichiers CSV
print(f"\n📥 Téléchargement des fichiers CSV depuis s3://{MINIO_BUCKET}/raw/...")

local_data_dir = '/opt/workspace/data'
os.makedirs(local_data_dir, exist_ok=True)

downloaded_files = []

try:
    response = s3_client.list_objects_v2(Bucket=MINIO_BUCKET, Prefix='raw/')
    
    if 'Contents' in response:
        for obj in response['Contents']:
            key = obj['Key']
            
            if key.endswith('.csv'):
                filename = os.path.basename(key)
                local_file = os.path.join(local_data_dir, filename)
                
                print(f"   📄 {filename}...", end=" ")
                s3_client.download_file(MINIO_BUCKET, key, local_file)
                
                size_mb = os.path.getsize(local_file) / (1024 * 1024)
                print(f"✓ ({size_mb:.2f} MB)")
                downloaded_files.append(local_file)
        
        print(f"\n✅ {len(downloaded_files)} fichier(s) téléchargé(s)")
    else:
        print("⚠️ Aucun fichier trouvé")
        sys.exit(1)
        
except Exception as e:
    print(f"\n❌ Erreur téléchargement: {e}")
    sys.exit(1)

# ============================================================================
# ÉTAPE 2: Traitement avec Spark (SANS S3A)
# ============================================================================
print("\n" + "="*70)
print("🚀 Initialisation de Spark")
print("="*70)

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, when, avg, sum as spark_sum

# Créer une session Spark simple (sans S3A)
spark = (
    SparkSession.builder
    .appName("TelcoChurnAnalysis")
    .master("local[*]")
    .config("spark.driver.memory", "2g")
    .config("spark.sql.adaptive.enabled", "true")
    .getOrCreate()
)

print(f"✅ Spark {spark.version} démarré")
print(f"   Master: {spark.sparkContext.master}")
print(f"   UI: http://localhost:4040")

# ============================================================================
# ÉTAPE 3: Charger les données avec Spark
# ============================================================================
print("\n📂 Chargement des données dans Spark...")

csv_pattern = os.path.join(local_data_dir, "*.csv")

try:
    df_raw = spark.read \
        .format("csv") \
        .option("header", "true") \
        .option("inferSchema", "true") \
        .option("mode", "DROPMALFORMED") \
        .load(csv_pattern)
    
    row_count = df_raw.count()
    col_count = len(df_raw.columns)
    
    print(f"✅ {row_count:,} lignes chargées")
    print(f"✅ {col_count} colonnes")
    
    # ========================================================================
    # ANALYSE DES DONNÉES
    # ========================================================================
    
    print("\n" + "="*70)
    print("📋 SCHÉMA DES DONNÉES")
    print("="*70)
    df_raw.printSchema()
    
    print("\n" + "="*70)
    print("📊 APERÇU DES DONNÉES (10 premières lignes)")
    print("="*70)
    df_raw.show(10, truncate=False)
    
    print("\n" + "="*70)
    print("📈 STATISTIQUES DESCRIPTIVES")
    print("="*70)
    df_raw.describe().show()
    
    # Analyse des valeurs nulles
    print("\n" + "="*70)
    print("🔍 ANALYSE DES VALEURS NULLES")
    print("="*70)
    
    null_counts = df_raw.select([
        count(when(col(c).isNull(), c)).alias(c) 
        for c in df_raw.columns
    ])
    
    null_data = null_counts.collect()[0].asDict()
    
    has_nulls = False
    print(f"{'Colonne':<30} {'Nulls':<10} {'%':<10}")
    print("-" * 50)
    
    for col_name, null_count in sorted(null_data.items()):
        if null_count > 0:
            pct = (null_count / row_count) * 100
            print(f"{col_name:<30} {null_count:<10} {pct:>6.2f}%")
            has_nulls = True
    
    if not has_nulls:
        print("✅ Aucune valeur nulle!")
    
    # Analyses spécifiques si colonnes connues
    if 'Churn' in df_raw.columns:
        print("\n" + "="*70)
        print("🎯 ANALYSE DU CHURN")
        print("="*70)
        
        print("\nDistribution du Churn:")
        df_raw.groupBy("Churn").count().orderBy("Churn").show()
        
        churn_yes = df_raw.filter(col("Churn") == "Yes").count()
        churn_no = df_raw.filter(col("Churn") == "No").count()
        churn_rate = (churn_yes / row_count) * 100
        
        print(f"📊 Taux de churn: {churn_rate:.2f}%")
        print(f"   • Churned: {churn_yes:,} clients ({churn_rate:.1f}%)")
        print(f"   • Retained: {churn_no:,} clients ({100-churn_rate:.1f}%)")
        
        # Analyse par tenure
        if 'tenure' in df_raw.columns:
            print("\n📅 Tenure moyenne par statut:")
            df_raw.groupBy("Churn") \
                .agg(avg("tenure").alias("Tenure_Moyenne")) \
                .orderBy("Churn") \
                .show()
        
        # Analyse par charges
        if 'MonthlyCharges' in df_raw.columns:
            print("\n💰 Charges mensuelles moyennes:")
            df_raw.groupBy("Churn") \
                .agg(avg("MonthlyCharges").alias("Charges_Moyennes")) \
                .orderBy("Churn") \
                .show()
        
        # Analyse par genre si disponible
        if 'gender' in df_raw.columns:
            print("\n👥 Distribution par genre:")
            df_raw.groupBy("gender", "Churn") \
                .count() \
                .orderBy("gender", "Churn") \
                .show()
    
    # ========================================================================
    # SAUVEGARDE DES RÉSULTATS
    # ========================================================================
    
    print("\n" + "="*70)
    print("💾 SAUVEGARDE DES DONNÉES")
    print("="*70)
    
    # 1. Sauvegarde locale en Parquet
    local_parquet = "/opt/workspace/telco_data.parquet"
    print(f"\n1️⃣ Sauvegarde Parquet local...")
    
    df_raw.write.mode("overwrite").parquet(local_parquet)
    print(f"   ✅ {local_parquet}")
    
    # Vérifier
    df_verify = spark.read.parquet(local_parquet)
    print(f"   ✅ Vérifié: {df_verify.count():,} lignes")
    
    # 2. Créer un fichier Parquet unique pour MinIO
    local_parquet_single = "/opt/workspace/telco_data_single.parquet"
    print(f"\n2️⃣ Création Parquet unique pour upload...")
    
    df_raw.coalesce(1).write.mode("overwrite").parquet(local_parquet_single)
    
    # Trouver le fichier part
    parquet_files = [f for f in os.listdir(local_parquet_single) 
                    if f.startswith('part-') and f.endswith('.parquet')]
    
    if parquet_files:
        local_file = os.path.join(local_parquet_single, parquet_files[0])
        file_size = os.path.getsize(local_file) / (1024 * 1024)
        print(f"   ✅ Fichier créé: {parquet_files[0]} ({file_size:.2f} MB)")
        
        # 3. Upload vers MinIO
        print(f"\n3️⃣ Upload vers MinIO...")
        
        try:
            s3_key = 'processed/telco_data.parquet'
            s3_client.upload_file(local_file, MINIO_BUCKET, s3_key)
            print(f"   ✅ Uploadé: s3://{MINIO_BUCKET}/{s3_key}")
            
            # Vérifier l'upload
            obj = s3_client.head_object(Bucket=MINIO_BUCKET, Key=s3_key)
            size_mb = obj['ContentLength'] / (1024 * 1024)
            print(f"   ✅ Taille sur MinIO: {size_mb:.2f} MB")
            
        except Exception as e:
            print(f"   ⚠️ Erreur upload: {e}")
    
    # 4. Sauvegarder aussi en CSV consolidé
    print(f"\n4️⃣ Création CSV consolidé...")
    
    local_csv = "/opt/workspace/telco_data_consolidated.csv"
    
    # Utiliser Pandas pour un CSV unique (plus simple)
    import pandas as pd
    df_pandas = df_raw.toPandas()
    df_pandas.to_csv(local_csv, index=False)
    
    csv_size = os.path.getsize(local_csv) / (1024 * 1024)
    print(f"   ✅ CSV créé: {local_csv} ({csv_size:.2f} MB)")
    
    # Upload CSV
    try:
        s3_client.upload_file(local_csv, MINIO_BUCKET, 'processed/telco_data_consolidated.csv')
        print(f"   ✅ CSV uploadé: s3://{MINIO_BUCKET}/processed/telco_data_consolidated.csv")
    except Exception as e:
        print(f"   ⚠️ Erreur upload CSV: {e}")
    
    # ========================================================================
    # RÉSUMÉ FINAL
    # ========================================================================
    
    print("\n" + "="*70)
    print("✅ TRAITEMENT TERMINÉ AVEC SUCCÈS!")
    print("="*70)
    
    print(f"\n📊 Statistiques:")
    print(f"   • Lignes traitées: {row_count:,}")
    print(f"   • Colonnes: {col_count}")
    print(f"   • Fichiers CSV source: {len(downloaded_files)}")
    
    print(f"\n📁 Fichiers générés:")
    print(f"   Local:")
    print(f"   • {local_parquet}")
    print(f"   • {local_csv}")
    print(f"   MinIO:")
    print(f"   • s3://{MINIO_BUCKET}/processed/telco_data.parquet")
    print(f"   • s3://{MINIO_BUCKET}/processed/telco_data_consolidated.csv")
    
    print(f"\n💡 Variables disponibles:")
    print(f"   • df_raw       : DataFrame Spark avec toutes les données")
    print(f"   • df_pandas    : DataFrame Pandas (pour analyses rapides)")
    print(f"   • spark        : Session Spark active")
    print(f"   • s3_client    : Client boto3 pour MinIO")
    
    print(f"\n📚 Exemples de commandes:")
    print(f"   df_raw.filter(col('Churn') == 'Yes').count()")
    print(f"   df_raw.groupBy('gender').count().show()")
    print(f"   df_pandas.describe()")
    
    print(f"\n🌐 Interfaces Web:")
    print(f"   • Spark UI: http://localhost:4040")
    print(f"   • MinIO Console: http://localhost:9001")
    
except Exception as e:
    print(f"\n❌ Erreur lors du traitement: {e}")
    import traceback
    traceback.print_exc()

print("\n" + "="*70)

🚀 TRAITEMENT TELCO CHURN - SPARK + MinIO

📦 Installation de boto3...



🔗 Connexion à MinIO...
✅ Connexion MinIO OK
   Buckets: ['telco-churn']

📥 Téléchargement des fichiers CSV depuis s3://telco-churn/raw/...
   📄 batch_1_1763820168.csv... ✓ (0.31 MB)
   📄 batch_2_1763820405.csv... ✓ (0.31 MB)
   📄 batch_3_1763820641.csv... ✓ (0.31 MB)

✅ 3 fichier(s) téléchargé(s)

🚀 Initialisation de Spark
✅ Spark 3.5.7 démarré
   Master: local[*]
   UI: http://localhost:4040

📂 Chargement des données dans Spark...


25/11/22 15:12:11 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


✅ 7,041 lignes chargées
✅ 21 colonnes

📋 SCHÉMA DES DONNÉES
root
 |-- customerID: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- SeniorCitizen: integer (nullable = true)
 |-- Partner: string (nullable = true)
 |-- Dependents: string (nullable = true)
 |-- tenure: integer (nullable = true)
 |-- PhoneService: string (nullable = true)
 |-- MultipleLines: string (nullable = true)
 |-- InternetService: string (nullable = true)
 |-- OnlineSecurity: string (nullable = true)
 |-- OnlineBackup: string (nullable = true)
 |-- DeviceProtection: string (nullable = true)
 |-- TechSupport: string (nullable = true)
 |-- StreamingTV: string (nullable = true)
 |-- StreamingMovies: string (nullable = true)
 |-- Contract: string (nullable = true)
 |-- PaperlessBilling: string (nullable = true)
 |-- PaymentMethod: string (nullable = true)
 |-- MonthlyCharges: double (nullable = true)
 |-- TotalCharges: string (nullable = true)
 |-- Churn: string (nullable = true)


📊 APERÇU DES DONNÉES

25/11/22 15:12:16 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

...

   ✅ /opt/workspace/telco_data.parquet
   ✅ Vérifié: 7,041 lignes

2️⃣ Création Parquet unique pour upload...
   ✅ Fichier créé: part-00000-c4fdea08-1eb6-4d90-ac35-76f862fbb35e-c000.snappy.parquet (0.17 MB)

3️⃣ Upload vers MinIO...
   ✅ Uploadé: s3://telco-churn/processed/telco_data.parquet
   ✅ Taille sur MinIO: 0.17 MB

4️⃣ Création CSV consolidé...



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/usr/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/usr/lib/python3.10/runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "/usr/local/lib/python3.10/dist-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/usr/local/lib/python3.10/dist-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/usr/local/lib/python3.10/dist-packages/ipykernel/kernelapp.p

AttributeError: _ARRAY_API not found


❌ Erreur lors du traitement: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject



Traceback (most recent call last):
  File "/tmp/ipykernel_271/4017085374.py", line 268, in <module>
    import pandas as pd
  File "/usr/local/lib/python3.10/dist-packages/pandas/__init__.py", line 46, in <module>
    from pandas.core.api import (
  File "/usr/local/lib/python3.10/dist-packages/pandas/core/api.py", line 1, in <module>
    from pandas._libs import (
  File "/usr/local/lib/python3.10/dist-packages/pandas/_libs/__init__.py", line 18, in <module>
    from pandas._libs.interval import Interval
  File "interval.pyx", line 1, in init pandas._libs.interval
ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject
